In [28]:
import os
import sys
import tempfile
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

sys.path.insert(0, os.path.dirname(os.getcwd()))

# load the dataset
DATA_PATH = os.path.join(tempfile.gettempdir(), "retail_store_inventory.csv")

In [29]:
# DATA_PATH

In [30]:
df = pd.read_csv(DATA_PATH)

In [31]:
df.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [32]:
df = df.drop_duplicates()
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')

In [33]:
df.shape

(73100, 15)

In [34]:
df['Date']

0       2022-01-01
1       2022-01-01
2       2022-01-01
3       2022-01-01
4       2022-01-01
           ...    
73095   2024-01-01
73096   2024-01-01
73097   2024-01-01
73098   2024-01-01
73099   2024-01-01
Name: Date, Length: 73100, dtype: datetime64[us]

In [35]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['quarter'] = df['Date'].dt.quarter
df['day_of_week'] = df['Date'].dt.dayofweek # Monday=0, Sunday=6
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int) # 1 for weekend, 0 for weekday
df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int) # Week number of the year (1-52)
df['days_since_start'] = (df['Date'] - df['Date'].min()).dt.days # Days since the earliest date in the dataset

print("Date features created successfully.")



Date features created successfully.


In [36]:
df

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,...,Competitor Pricing,Seasonality,Year,Month,Day,quarter,day_of_week,is_weekend,week_of_year,days_since_start
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,...,29.69,Autumn,2022,1,1,1,5,1,52,0
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,...,66.16,Autumn,2022,1,1,1,5,1,52,0
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,...,31.32,Summer,2022,1,1,1,5,1,52,0
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,...,34.74,Autumn,2022,1,1,1,5,1,52,0
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,...,68.95,Summer,2022,1,1,1,5,1,52,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73095,2024-01-01,S005,P0016,Furniture,East,96,8,127,18.46,73.73,...,72.45,Winter,2024,1,1,1,0,0,1,730
73096,2024-01-01,S005,P0017,Toys,North,313,51,101,48.43,82.57,...,83.78,Autumn,2024,1,1,1,0,0,1,730
73097,2024-01-01,S005,P0018,Clothing,West,278,36,151,39.65,11.11,...,10.91,Winter,2024,1,1,1,0,0,1,730
73098,2024-01-01,S005,P0019,Toys,East,374,264,21,270.52,53.14,...,55.80,Spring,2024,1,1,1,0,0,1,730


In [37]:
group = df.groupby(["Store ID", "Product ID"])["Demand Forecast"]

In [38]:
df['demand_lag_7'] = group.shift(7) # Lag of 7 days (1 week)
df['demand_lag_14'] = group.shift(14) # Lag of 14 days (2 weeks)
df['demand_lag_30'] = group.shift(30) # Lag of 30 days (1 month)
df['demand_rolling_mean_7'] = group.transform(lambda x: x.shift(1).rolling(7).mean()) # 7-day rolling mean
df['demand_rolling_mean_14'] = group.transform(lambda x: x.shift(1).rolling(14).mean()) # 14-day rolling mean
df['demand_rolling_mean_30'] = group.transform(lambda x: x.shift(1).rolling(30).mean()) # 30-day rolling mean


rows_before = len(df)

df = df.dropna(subset=['demand_lag_7', 'demand_lag_14', 'demand_lag_30', 'demand_rolling_mean_7', 'demand_rolling_mean_14', 'demand_rolling_mean_30'])

rows_after = len(df)
print(f"Rows removed: {rows_before - rows_after}")

Rows removed: 3000


In [39]:
rows_after

70100

In [40]:
df

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,...,day_of_week,is_weekend,week_of_year,days_since_start,demand_lag_7,demand_lag_14,demand_lag_30,demand_rolling_mean_7,demand_rolling_mean_14,demand_rolling_mean_30
3000,2022-01-31,S001,P0001,Clothing,West,410,200,152,212.24,70.78,...,0,0,5,30,17.87,73.59,135.47,106.825714,137.663571,119.560000
3001,2022-01-31,S001,P0002,Electronics,South,411,363,63,375.34,80.69,...,0,0,5,30,152.76,169.21,144.04,114.037143,144.236429,151.721667
3002,2022-01-31,S001,P0003,Clothing,East,126,94,176,86.82,11.21,...,0,0,5,30,143.44,16.69,74.02,181.832857,127.355714,118.996667
3003,2022-01-31,S001,P0004,Toys,East,113,105,152,98.38,39.90,...,0,0,5,30,77.48,99.26,62.18,160.178571,161.297143,148.466000
3004,2022-01-31,S001,P0005,Electronics,West,368,126,132,129.14,28.06,...,0,0,5,30,95.99,74.40,9.26,169.868571,157.407857,150.310333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73095,2024-01-01,S005,P0016,Furniture,East,96,8,127,18.46,73.73,...,0,0,1,730,50.31,235.99,351.56,95.588571,126.856429,150.415000
73096,2024-01-01,S005,P0017,Toys,North,313,51,101,48.43,82.57,...,0,0,1,730,115.90,280.00,206.14,143.122857,138.905000,136.092000
73097,2024-01-01,S005,P0018,Clothing,West,278,36,151,39.65,11.11,...,0,0,1,730,-4.57,65.61,76.13,97.531429,111.190714,110.581667
73098,2024-01-01,S005,P0019,Toys,East,374,264,21,270.52,53.14,...,0,0,1,730,206.97,284.12,229.15,138.820000,141.940000,148.009667


In [41]:
df['discount_rate'] = df['Discount'] / df['Price'].replace(0, np.nan) # Avoid division by zero

In [42]:
df['sell_through_rate'] = df['Units Sold'] / df['Inventory Level'].replace(0, np.nan) # Avoid division by zero

In [43]:
df['supply_gap'] = df['Units Ordered'] - df['Units Sold']

In [44]:
df['supply_gap']

3000     -48
3001    -300
3002      82
3003      47
3004       6
        ... 
73095    119
73096     50
73097    115
73098   -243
73099    159
Name: supply_gap, Length: 70100, dtype: int64

In [45]:
df['price_vs_competitor'] = df['Price'] - df['Competitor Pricing']

In [46]:
df['price_vs_competitor']

3000     1.90
3001     4.67
3002     1.12
3003     2.35
3004     2.44
         ... 
73095    1.28
73096   -1.21
73097    0.20
73098   -2.66
73099   -1.13
Name: price_vs_competitor, Length: 70100, dtype: float64

In [47]:
df

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,...,demand_lag_7,demand_lag_14,demand_lag_30,demand_rolling_mean_7,demand_rolling_mean_14,demand_rolling_mean_30,discount_rate,sell_through_rate,supply_gap,price_vs_competitor
3000,2022-01-31,S001,P0001,Clothing,West,410,200,152,212.24,70.78,...,17.87,73.59,135.47,106.825714,137.663571,119.560000,0.000000,0.487805,-48,1.90
3001,2022-01-31,S001,P0002,Electronics,South,411,363,63,375.34,80.69,...,152.76,169.21,144.04,114.037143,144.236429,151.721667,0.000000,0.883212,-300,4.67
3002,2022-01-31,S001,P0003,Clothing,East,126,94,176,86.82,11.21,...,143.44,16.69,74.02,181.832857,127.355714,118.996667,0.446030,0.746032,82,1.12
3003,2022-01-31,S001,P0004,Toys,East,113,105,152,98.38,39.90,...,77.48,99.26,62.18,160.178571,161.297143,148.466000,0.375940,0.929204,47,2.35
3004,2022-01-31,S001,P0005,Electronics,West,368,126,132,129.14,28.06,...,95.99,74.40,9.26,169.868571,157.407857,150.310333,0.534569,0.342391,6,2.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73095,2024-01-01,S005,P0016,Furniture,East,96,8,127,18.46,73.73,...,50.31,235.99,351.56,95.588571,126.856429,150.415000,0.271260,0.083333,119,1.28
73096,2024-01-01,S005,P0017,Toys,North,313,51,101,48.43,82.57,...,115.90,280.00,206.14,143.122857,138.905000,136.092000,0.121109,0.162939,50,-1.21
73097,2024-01-01,S005,P0018,Clothing,West,278,36,151,39.65,11.11,...,-4.57,65.61,76.13,97.531429,111.190714,110.581667,0.900090,0.129496,115,0.20
73098,2024-01-01,S005,P0019,Toys,East,374,264,21,270.52,53.14,...,206.97,284.12,229.15,138.820000,141.940000,148.009667,0.376364,0.705882,-243,-2.66


In [48]:
# Label encode categorical variables
# A - 0, B - 1, C - 2, D - 3, E - 4


# clothing - 1,
# Toys - 2,
# Electronics - 3,
# Furniture - 4
# groceries - 5

label_cols = ["Store ID", "Product ID", "Category", "Region", "Weather Condition", "Seasonality"]

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
le = LabelEncoder()
encoders = {}
for col in label_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = list(le.classes_)






In [49]:
# df.info()

In [50]:
print(encoders)

{'Store ID': ['S001', 'S002', 'S003', 'S004', 'S005'], 'Product ID': ['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008', 'P0009', 'P0010', 'P0011', 'P0012', 'P0013', 'P0014', 'P0015', 'P0016', 'P0017', 'P0018', 'P0019', 'P0020'], 'Category': ['Clothing', 'Electronics', 'Furniture', 'Groceries', 'Toys'], 'Region': ['East', 'North', 'South', 'West'], 'Weather Condition': ['Cloudy', 'Rainy', 'Snowy', 'Sunny'], 'Seasonality': ['Autumn', 'Spring', 'Summer', 'Winter']}


In [51]:
df

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,...,demand_lag_7,demand_lag_14,demand_lag_30,demand_rolling_mean_7,demand_rolling_mean_14,demand_rolling_mean_30,discount_rate,sell_through_rate,supply_gap,price_vs_competitor
3000,2022-01-31,0,0,0,3,410,200,152,212.24,70.78,...,17.87,73.59,135.47,106.825714,137.663571,119.560000,0.000000,0.487805,-48,1.90
3001,2022-01-31,0,1,1,2,411,363,63,375.34,80.69,...,152.76,169.21,144.04,114.037143,144.236429,151.721667,0.000000,0.883212,-300,4.67
3002,2022-01-31,0,2,0,0,126,94,176,86.82,11.21,...,143.44,16.69,74.02,181.832857,127.355714,118.996667,0.446030,0.746032,82,1.12
3003,2022-01-31,0,3,4,0,113,105,152,98.38,39.90,...,77.48,99.26,62.18,160.178571,161.297143,148.466000,0.375940,0.929204,47,2.35
3004,2022-01-31,0,4,1,3,368,126,132,129.14,28.06,...,95.99,74.40,9.26,169.868571,157.407857,150.310333,0.534569,0.342391,6,2.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73095,2024-01-01,4,15,2,0,96,8,127,18.46,73.73,...,50.31,235.99,351.56,95.588571,126.856429,150.415000,0.271260,0.083333,119,1.28
73096,2024-01-01,4,16,4,1,313,51,101,48.43,82.57,...,115.90,280.00,206.14,143.122857,138.905000,136.092000,0.121109,0.162939,50,-1.21
73097,2024-01-01,4,17,0,3,278,36,151,39.65,11.11,...,-4.57,65.61,76.13,97.531429,111.190714,110.581667,0.900090,0.129496,115,0.20
73098,2024-01-01,4,18,4,0,374,264,21,270.52,53.14,...,206.97,284.12,229.15,138.820000,141.940000,148.009667,0.376364,0.705882,-243,-2.66


In [52]:
df = df.drop(columns=["Date"])

In [53]:
df

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,...,demand_lag_7,demand_lag_14,demand_lag_30,demand_rolling_mean_7,demand_rolling_mean_14,demand_rolling_mean_30,discount_rate,sell_through_rate,supply_gap,price_vs_competitor
3000,0,0,0,3,410,200,152,212.24,70.78,0,...,17.87,73.59,135.47,106.825714,137.663571,119.560000,0.000000,0.487805,-48,1.90
3001,0,1,1,2,411,363,63,375.34,80.69,0,...,152.76,169.21,144.04,114.037143,144.236429,151.721667,0.000000,0.883212,-300,4.67
3002,0,2,0,0,126,94,176,86.82,11.21,5,...,143.44,16.69,74.02,181.832857,127.355714,118.996667,0.446030,0.746032,82,1.12
3003,0,3,4,0,113,105,152,98.38,39.90,15,...,77.48,99.26,62.18,160.178571,161.297143,148.466000,0.375940,0.929204,47,2.35
3004,0,4,1,3,368,126,132,129.14,28.06,15,...,95.99,74.40,9.26,169.868571,157.407857,150.310333,0.534569,0.342391,6,2.44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73095,4,15,2,0,96,8,127,18.46,73.73,20,...,50.31,235.99,351.56,95.588571,126.856429,150.415000,0.271260,0.083333,119,1.28
73096,4,16,4,1,313,51,101,48.43,82.57,10,...,115.90,280.00,206.14,143.122857,138.905000,136.092000,0.121109,0.162939,50,-1.21
73097,4,17,0,3,278,36,151,39.65,11.11,10,...,-4.57,65.61,76.13,97.531429,111.190714,110.581667,0.900090,0.129496,115,0.20
73098,4,18,4,0,374,264,21,270.52,53.14,20,...,206.97,284.12,229.15,138.820000,141.940000,148.009667,0.376364,0.705882,-243,-2.66


In [54]:
from google.cloud import storage
from settings.setting import processed_data_path, SERVICE_ACCOUNT_KEY_PATH, BUCKET_NAME

LOCAL_PARQUET = os.path.join(tempfile.gettempdir(), "supply_chain_data_processed.parquet")
GCS_BLOB_NAME = "demand_forecast_processed_data/supply_chain_data_processed.parquet"

df.to_parquet(LOCAL_PARQUET, index=False, engine="pyarrow")


In [56]:
# upload the parquet into GCS
storage_client = storage.Client.from_service_account_json(SERVICE_ACCOUNT_KEY_PATH)
bucket = storage_client.bucket(bucket_name=BUCKET_NAME)
blob = bucket.blob(GCS_BLOB_NAME)
blob.upload_from_filename(LOCAL_PARQUET)

gcs_uri = f"gs://{BUCKET_NAME}/{GCS_BLOB_NAME}"
print(gcs_uri)

gs://machine_learning_datasets/demand_forecast_processed_data/supply_chain_data_processed.parquet


In [59]:
df_verify = pd.read_parquet(gcs_uri, storage_options={"token": SERVICE_ACCOUNT_KEY_PATH})
df_verify.head()

,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,...,demand_lag_7,demand_lag_14,demand_lag_30,demand_rolling_mean_7,demand_rolling_mean_14,demand_rolling_mean_30,discount_rate,sell_through_rate,supply_gap,price_vs_competitor
0,0,0,0,3,410,200,152,212.24,70.78,0,...,17.87,73.59,135.47,106.825714,137.663571,119.560000,0.000000,0.487805,-48,1.90
1,0,1,1,2,411,363,63,375.34,80.69,0,...,152.76,169.21,144.04,114.037143,144.236429,151.721667,0.000000,0.883212,-300,4.67
2,0,2,0,0,126,94,176,86.82,11.21,5,...,143.44,16.69,74.02,181.832857,127.355714,118.996667,0.446030,0.746032,82,1.12
3,0,3,4,0,113,105,152,98.38,39.90,15,...,77.48,99.26,62.18,160.178571,161.297143,148.466000,0.375940,0.929204,47,2.35
4,0,4,1,3,368,126,132,129.14,28.06,15,...,95.99,74.40,9.26,169.868571,157.407857,150.310333,0.534569,0.342391,6,2.44
